In [ ]:
import os

width, height = 1280, 720


def get_crop_coords(path):
    split = path.split(os.sep)
    if len(split) < 4:
        raise ValueError()

    year = int(split[-4])
    month = int(split[-3])

    if year > 2023 or (year == 2023 and month == 12):
        x_start, x_end = 966, 1216
        y_start, y_end = 287, 510
    else:
        x_start, x_end = 1018, 1210
        y_start, y_end = 474, 680

    return x_start, x_end, y_start, y_end

In [ ]:
import numpy as np
import torch
from PIL import Image
from torchvision import models
from torchvision import transforms as T
from torchvision.transforms import functional as F

TARGET_W, TARGET_H = 250, 223


def pad_to_target(img):
    w, h = img.size  # PIL order: (w, h)

    pad_width = TARGET_W - w
    pad_height = TARGET_H - h

    pad_l = pad_width // 2
    pad_r = pad_width - pad_l
    pad_t = pad_height // 2
    pad_b = pad_height - pad_t

    # F.pad expects (left, top, right, bottom)
    return F.pad(img, padding=[pad_l, pad_t, pad_r, pad_b], fill=0)


def load_interpreter_model(model_path: str):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model = models.mobilenet_v3_small()
    model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, 2)  # type: ignore
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval().to(device)

    # ImageNet defaults (safe even if weights_enum.meta is absent)
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)

    transform = T.Compose(
        [
            T.Lambda(pad_to_target),
            T.ToTensor(),
            T.Normalize(mean, std),
        ]
    )
    return model, transform, device


@torch.inference_mode()
def has_interpreter_batch(
    frames_bgr: list[np.ndarray] | np.ndarray,
    model: torch.nn.Module,
    transform: T.Compose,
    device: torch.device | str = 'cpu',
    threshold: float = 0.5,
    return_probs: bool = False,
):
    if isinstance(frames_bgr, np.ndarray) and frames_bgr.ndim == 4:
        frames_iter = frames_bgr
    else:
        frames_iter = list(frames_bgr)

    tensors = []
    for frm in frames_iter:  # BGR ➜ RGB ➜ tensor
        img_rgb = frm[..., ::-1]
        pil = Image.fromarray(img_rgb)
        tensors.append(transform(pil))
    batch = torch.stack(tensors).to(device)

    logits = model(batch)
    probs = torch.softmax(logits, dim=1)[:, 1]  # class-1 = interpreter
    preds = (probs >= threshold).cpu().tolist()

    if return_probs:
        return preds, probs.cpu().tolist()
    return preds


In [ ]:
import json

import cv2

fps = 25
model, tfm, dev = load_interpreter_model('interpreter_head.pt')


def has_interpreter_generator(video_path, batch_size=1024):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return

    x_start, x_end, y_start, y_end = get_crop_coords(video_path)

    done = False
    while not done:
        frames_batch = []

        for _ in range(batch_size):
            ok, frame = cap.read()
            if not ok:
                done = True
                break

            frame = frame[y_start:y_end, x_start:x_end]
            frames_batch.append(frame)

        if not frames_batch:
            break

        results = has_interpreter_batch(frames_batch, model, tfm, dev)
        yield from results

    cap.release()


def process(video_path: str):
    # --- State and Threshold Initialization ---
    segments = []
    current_segment_start = -1
    disappearance_start_frame = -1
    potential_presence_start_frame = -1

    # Thresholds are clearly defined
    disappearance_threshold_frames = int(2 * fps)
    long_absence_threshold_frames = int(90 * fps)
    min_duration_frames = int(10 * fps)
    presence_confirmation_frames = int(2 * fps)

    # --- Main Processing Loop ---
    for idx, has_interpreter in enumerate(has_interpreter_generator(video_path)):
        if not has_interpreter:
            # If no interpreter is detected, any potential presence is immediately voided.
            potential_presence_start_frame = -1

            # If we were in a confirmed segment, start tracking the disappearance.
            if current_segment_start >= 0 and disappearance_start_frame < 0:
                disappearance_start_frame = idx

            if disappearance_start_frame >= 0:
                # Determine if the disappearance is long enough to end the segment
                is_disappearance_confirmed = (
                    idx - disappearance_start_frame
                ) >= disappearance_threshold_frames
                is_long_absence = (
                    idx - disappearance_start_frame
                ) >= long_absence_threshold_frames

                if is_disappearance_confirmed:
                    segment_end = disappearance_start_frame - 1

                    # Validate the segment's duration before adding it
                    if (segment_end - current_segment_start) >= min_duration_frames:
                        segments.append(
                            {'start': current_segment_start, 'end': segment_end}
                        )

                    # Reset the segment state
                    current_segment_start = -1
                    disappearance_start_frame = -1

                    # If the absence is long, stop processing the rest of the video
                    if is_long_absence:
                        print(
                            f'Long absence detected at frame {idx}. Halting analysis.'
                        )
                        break
            continue  # Move to the next frame

        # This block runs only if has_interpreter is True
        if current_segment_start >= 0:
            # If we are already in a confirmed segment, just reset the disappearance timer
            disappearance_start_frame = -1
        else:
            # Otherwise, manage the potential presence confirmation
            if potential_presence_start_frame < 0:
                potential_presence_start_frame = idx

            # Check if the potential presence has now been confirmed
            if (idx - potential_presence_start_frame) >= presence_confirmation_frames:
                current_segment_start = potential_presence_start_frame
                disappearance_start_frame = -1

    # --- Finalization ---
    # After the loop, check if there's a valid, unterminated segment and save it.
    # This handles the case where the video ends while an interpreter is present.
    if current_segment_start >= 0:
        # The last processed frame's index is `idx`. If the loop finished, it processed up to `idx`.
        # Therefore, the segment ends at the last frame index.
        last_frame_index = idx
        segment_end = last_frame_index

        if (segment_end - current_segment_start) >= min_duration_frames:
            segments.append({'start': current_segment_start, 'end': segment_end})

    # --- Save Results ---
    with open(video_path.replace('video.mp4', 'segments.json'), 'w') as f:
        json.dump(segments, f, indent=2)

In [ ]:
for root, _, files in os.walk('.'):
    for file in files:
        if file == '.has_interpreter':
            video = os.path.join(root, 'video.mp4')
            if os.path.exists(video.replace('video.mp4', 'segments.json')):
                print(f'skipping {video}')
                continue
            process(video)
            print(f'processed {video}')

In [ ]:
###### Actually crop with ffmpeg ######

import json  # noqa: F811
import subprocess

fps = 25


def crop_segments(root: str):
    split = root.split(os.sep)
    year = int(split[-3])
    month = int(split[-2])

    video = os.path.join(root, 'video.mp4')
    x_start, x_end, y_start, y_end = get_crop_coords(video)

    if year > 2023 or (year == 2023 and month == 12):
        top, bottom = y_start + 1, height - y_end - 1
        left, right = x_start + 2, width - x_end - 0
    else:
        top, bottom = y_start + 2, height - y_end - 1
        left, right = x_start + 2, width - x_end - 1

    with open(os.path.join(root, 'segments.json')) as f:
        segments = json.load(f)

    for i, seg in enumerate(segments):
        s_frame, e_frame = seg['start'], seg['end']
        s_sec, e_sec = s_frame / fps, e_frame / fps

        filters = ';'.join(
            [
                f'[0:v]trim=start_frame={s_frame}:end_frame={e_frame},setpts=PTS-STARTPTS[vtmp]',
                '[vtmp]setsar=1[v]',
                f'[0:a]atrim=start={s_sec}:end={e_sec},asetpts=PTS-STARTPTS[a]',
            ]
        )

        out = os.path.join(root, f'segment_{i}.mp4')

        subprocess.run(
            [
                'ffmpeg',
                '-vsync',
                '0',
                '-hwaccel',
                'cuda',
                '-c:v',
                'h264_cuvid',
                '-crop',
                f'{top}x{bottom}x{left}x{right}',
                '-i',
                video,
                '-filter_complex',
                filters,
                '-map',
                '[v]',
                '-c:v',
                'h264_nvenc',
                '-preset',
                'p1',
                '-map',
                '[a]',
                '-c:a',
                'aac',
                '-b:a',
                '128k',
                out,
            ],
            check=True,
        )

In [ ]:
import os
import traceback

todo = []


for root, _, files in os.walk('.'):
    for file in files:
        if file == 'segments.json':
            if os.path.exists(os.path.join(root, 'segment_0.mp4')):
                print(f'skipping {video}')
                continue
            todo.append(root)

for r in todo:
    try:
        crop_segments(r)
        print(f'✓ processed  {r}')
    except Exception:
        print(f'✗ FAILED     {r}\n{traceback.format_exc()}')

In [ ]:
import os
import re

import numpy as np

SEGMENT_RE = re.compile(r'^segment_(\d+)\.mp4$')

for root, _, files in os.walk('.'):
    for file in files:
        m = SEGMENT_RE.match(file)
        if not m:
            continue

        idx = m.group(1)
        out_path = os.path.join(root, f'mean_{idx}.png')
        if os.path.exists(out_path):
            print(f'Skipping {file}')
            continue

        segment_path = os.path.join(root, file)
        cap = cv2.VideoCapture(segment_path)
        if not cap.isOpened():
            continue

        mean = None
        count = 0

        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame = frame.astype(np.float64)
            if mean is None:
                mean = frame
            else:
                mean += frame
            count += 1

        cap.release()
        if count == 0:
            print(f'No frames in {file}')
            continue

        mean /= count
        mean = mean.clip(0, 255).astype(np.uint8)
        cv2.imwrite(out_path, mean)
        print(f'Wrote mean frame → {out_path}')